# Fast R-CNN Implementation
**Assignment 3 - Reproducing "Fast R-CNN" (Ross Girshick, 2015)**

This notebook implements the Fast R-CNN object detection architecture from scratch.
It performs the following:
1. Downloads and prepares the **PennFudanPed** pedestrian detection dataset.
2. Implements a custom **Fast R-CNN** network combining a VGG16 feature extractor, an **RoI Pooling** layer, and a multi-task (classification + bounding box regression) head.
3. Rapidly trains using a custom RoI Proposal Generator (simulating Selective Search).
4. Computes the Multi-task Loss ($L_{cls} + \lambda L_{loc}$).
5. Visualizes bounding box predictions using Matplotlib.


In [ ]:
import os
import zipfile
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torchvision.models import vgg16, VGG16_Weights
from torchvision.ops import roi_pool, box_iou
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import xml.etree.ElementTree as ET
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Detect hardware
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")



## 1. Dataset Preparation (PennFudanPed)
We use a lightweight object detection dataset with ground truth bounding boxes mapping to pedestrians.


In [ ]:
# Download PennFudan dataset
url = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"
download_path = "PennFudanPed.zip"
extract_path = "./"

if not os.path.exists("./PennFudanPed"):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, download_path)
    with zipfile.ZipFile(download_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    os.remove(download_path)
    print("Downloaded and extracted PennFudanPed.")

class PennFudanDataset(Dataset):
    def __init__(self, root):
        self.root = root
        self.imgs = list(sorted(os.listdir(os.path.join(root, "PNGImages"))))
        self.masks = list(sorted(os.listdir(os.path.join(root, "PedMasks"))))

    def __getitem__(self, idx):
        # Load images and masks
        img_path = os.path.join(self.root, "PNGImages", self.imgs[idx])
        mask_path = os.path.join(self.root, "PedMasks", self.masks[idx])
        
        # Fast R-CNN standardizes images, we will just use a basic transform
        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)
        
        # Convert mask to numpy to get bbox
        mask = np.array(mask)
        obj_ids = np.unique(mask)
        obj_ids = obj_ids[1:] # discard background
        
        boxes = []
        for i in obj_ids:
            pos = np.where(mask == i)
            xmin = np.min(pos[1])
            xmax = np.max(pos[1])
            ymin = np.min(pos[0])
            ymax = np.max(pos[0])
            if xmax > xmin and ymax > ymin:
                boxes.append([xmin, ymin, xmax, ymax])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        # Class 1 is pedestrian, 0 is background
        labels = torch.ones((len(boxes),), dtype=torch.int64) 
        img_tensor = torchvision.transforms.functional.to_tensor(img)
        img_tensor = torchvision.transforms.functional.normalize(img_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        
        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        
        return img_tensor, target

    def __len__(self):
        return len(self.imgs)

# Custom collate for object detection batching
def collate_fn(batch):
    return tuple(zip(*batch))

dataset = PennFudanDataset('PennFudanPed')
torch.manual_seed(1)
indices = torch.randperm(len(dataset)).tolist()
dataset_train = torch.utils.data.Subset(dataset, indices[:-50])
dataset_test = torch.utils.data.Subset(dataset, indices[-50:])

train_loader = DataLoader(dataset_train, batch_size=1, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(dataset_test, batch_size=1, shuffle=False, collate_fn=collate_fn)



## 2. Fast R-CNN Architecture
The core innovation of Fast R-CNN is the **RoI Pooling** layer. Instead of running the massive CNN on 2000 external bounding boxes separately (like standard R-CNN), Fast R-CNN runs the CNN once over the entire image. 

Given bounding box region proposals (RoIs), it pulls those localized sections out of the dense feature map via RoIPooling, flattening them sequentially through two heads: `cls_score` and `bbox_pred`.


In [ ]:
class FastRCNN(nn.Module):
    def __init__(self, num_classes):
        super(FastRCNN, self).__init__()
        # 1. Feature Extractor (VGG16 as used in the paper)
        vgg = vgg16(weights=VGG16_Weights.DEFAULT)
        # We strip the last MaxPool to keep the feature map slightly larger
        self.features = nn.Sequential(*list(vgg.features.children())[:-1]) 
        
        # 2. RoI Pooling layer (downsamples any size box to exactly 7x7)
        # spatial_scale = 1/16 because VGG downsamples exactly 16x (4 maxpool layers of 1/2)
        self.roi_pool = torchvision.ops.RoIPool(output_size=(7, 7), spatial_scale=1./16.)
        
        # 3. Dense Classifier (the fully connected layers of VGG)
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
        )
        
        # 4. Multi-Task Sibling Heads
        self.cls_score = nn.Linear(4096, num_classes)
        self.bbox_pred = nn.Linear(4096, num_classes * 4)

    def forward(self, images, rois):
        """
        images: Batch of images (B, C, H, W)
        rois: List of bounding boxes generated by a proposal algorithm. 
              Tensor shape per image: (N, 4) coordinates [x1, y1, x2, y2]
        """
        # Convert RoIs to the format expected by roi_pool: [batch_index, x1, y1, x2, y2]
        roi_list = []
        for i, img_rois in enumerate(rois):
            batch_idx = torch.full((img_rois.shape[0], 1), i, device=img_rois.device, dtype=torch.float)
            roi_list.append(torch.cat([batch_idx, img_rois], dim=1))
        
        pooled_rois = torch.cat(roi_list, dim=0)

        # 1. Extract entire image features
        base_features = self.features(images)
        
        # 2. Extract specific RoI pools
        pooled_features = self.roi_pool(base_features, pooled_rois)
        
        # 3. Flatten and pass to classifier
        x = pooled_features.view(pooled_features.size(0), -1)
        x = self.classifier(x)
        
        # 4. output classifications and regression
        cls_logits = self.cls_score(x)
        bbox_regs = self.bbox_pred(x)
        
        return cls_logits, bbox_regs



## 3. Proposal Simulator & Multi-Task Loss Function
The Fast R-CNN paper relies on *Selective Search* to generate 2000 box proposals per image. To train our model natively in PyTorch without a third-party C++ Selective Search library, we will generate "simulated proposals" by taking the Ground Truth boxes (Positives) and scattering randomly generated bad boxes around the image (Negatives/Background class 0).

The multi-task loss is:
$$ L(p, u, t^u, v) = L_{cls}(p, u) + \lambda[u \ge 1] L_{loc}(t^u, v) $$



In [ ]:
# Helper to generate simulated proposals for training
def generate_simulated_proposals(targets, img_shape, num_negatives=16):
    rois_list = []
    labels_list = []
    
    for t in targets:
        gt_boxes = t["boxes"]
        
        # Start with ground truth (Positive RoIs -> Label 1)
        proposals = gt_boxes.clone()
        labels = torch.ones(len(gt_boxes), dtype=torch.int64, device=gt_boxes.device)
        
        # Generate random Negative RoIs (Label 0)
        neg_boxes = []
        H, W = img_shape
        for _ in range(num_negatives):
            x1 = torch.rand(1) * W
            y1 = torch.rand(1) * H
            # Ensure box has some size
            x2 = x1 + torch.rand(1) * (W - x1)
            y2 = y1 + torch.rand(1) * (H - y1)
            
            # Simple check to avoid degenerate boxes
            if x2 - x1 >= 10 and y2 - y1 >= 10:
                neg_boxes.append(torch.tensor([x1.item(), y1.item(), x2.item(), y2.item()], device=gt_boxes.device))
        
        if len(neg_boxes) > 0:
            neg_boxes = torch.stack(neg_boxes)
            proposals = torch.cat([proposals, neg_boxes], dim=0)
            labels = torch.cat([labels, torch.zeros(len(neg_boxes), dtype=torch.int64, device=gt_boxes.device)], dim=0)
            
        rois_list.append(proposals)
        labels_list.append(labels)
        
    return rois_list, labels_list

def fast_rcnn_loss(cls_logits, bbox_regs, labels, gt_boxes_concat):
    # 1. Classification Loss (Cross Entropy)
    loss_cls = F.cross_entropy(cls_logits, labels)
    
    # 2. Bounding Box Regression Loss (Smooth L1 Loss)
    # We only compute bbox loss for positive classes (label >= 1)
    pos_idx = labels > 0
    if pos_idx.sum() > 0:
        pos_bbox_regs = bbox_regs[pos_idx]
        pos_gt_boxes = gt_boxes_concat[pos_idx]
        
        # For multi-class, bbox_regs has shape (N, num_classes*4). We must extract the specific class coordinates.
        # But we only have 1 foreground class (index 1), so coords are at index 4:8
        pos_bbox_regs = pos_bbox_regs[:, 4:8] 
        loss_loc = F.smooth_l1_loss(pos_bbox_regs, pos_gt_boxes, reduction='mean')
    else:
        loss_loc = torch.tensor(0.0).to(cls_logits.device)
        
    return loss_cls + loss_loc



## 4. Training Loop


In [ ]:
num_classes = 2 # 1 class (Person) + 1 Background (0)
model = FastRCNN(num_classes).to(device)

# Optimizers config specified heavily in the paper
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(params, lr=1e-4, momentum=0.9, weight_decay=0.0005)

epochs = 5 # Decrease for rapid toy testing

print("Starting Fast R-CNN Training...")
model.train()

loss_history = []

for epoch in range(epochs):
    running_loss = 0.0
    for i, (images, targets) in enumerate(train_loader):
        # Move images and targets to device
        images_tensor = torch.stack(images).to(device)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        optimizer.zero_grad()
        
        # Simulate selective search proposals (GT + Negative Backgrounds)
        rois, labels_list = generate_simulated_proposals(targets, (images_tensor.shape[2], images_tensor.shape[3]))
        
        # Forward pass!
        cls_logits, bbox_regs = model(images_tensor, rois)
        
        # Flatten labels and gt_boxes to match logits shapes
        flat_labels = torch.cat(labels_list, dim=0)
        # Note: Bbox regression targets in paper are normalized parameterized (dx, dy). 
        # For simplicity in this toy implementation, we regress absolute coordinates.
        flat_gt_boxes = torch.cat([t["boxes"] for t in targets], dim=0)
        # Pad flat_gt_boxes with zeros for backgrounds so lengths align
        padded_gt_boxes = torch.zeros((flat_labels.shape[0], 4), device=device)
        total_gt = sum(len(t["boxes"]) for t in targets)
        padded_gt_boxes[:total_gt] = flat_gt_boxes
        
        loss = fast_rcnn_loss(cls_logits, bbox_regs, flat_labels, padded_gt_boxes)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_loss = running_loss / len(train_loader)
    loss_history.append(avg_loss)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f}")

print("Training finished.")

plt.plot(loss_history)
plt.title("Fast R-CNN Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()



## 5. Visualizing Detection Outputs
Let's see what the network predicted! Fast R-CNN traditionally relies on an external proposal algorithm (like Selective Search). To demonstrate the power of our bounding box regressor, we will feed the network a deliberately noisy "bad" proposal, and watch the network's Regression Head correct it to tightly frame the pedestrian!


In [ ]:
model.eval()

# Grab a single test image
images, targets = next(iter(test_loader))
img_tensor = images[0].unsqueeze(0).to(device)

# Simulate a Selective Search region proposal by taking the ground truth and displacing it
gt_box = targets[0]["boxes"][0].cpu()
noisy_proposal = gt_box + torch.tensor([-30., -30., 40., 40.]) # heavily expanded/shifted box
test_rois = noisy_proposal.unsqueeze(0).to(device)

with torch.no_grad():
    cls_logits, bbox_regs = model(img_tensor, [test_rois])
    probs = F.softmax(cls_logits, dim=1)
    
    # Grab the network's absolute coordinate prediction for the 'Person' class (index 4 to 8)
    final_box = bbox_regs[0, 4:8].cpu().numpy()

# Plotting the image
fig, ax = plt.subplots(1, figsize=(8, 8))
img_for_plot = images[0].permute(1, 2, 0).numpy()

# Un-normalize the ImageNet scaling to display correct colors
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
img_for_plot = std * img_for_plot + mean
img_for_plot = np.clip(img_for_plot, 0, 1)

ax.imshow(img_for_plot)

# Overlay the INITIAL noisy proposal (Yellow Dashed)
x1, y1, x2, y2 = noisy_proposal.numpy()
rect1 = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='yellow', linestyle='dashed', facecolor='none')
ax.add_patch(rect1)

# Overlay the NETWORK PREDICTED bounding box (Solid Red)
x1, y1, x2, y2 = final_box
rect2 = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='red', facecolor='none')
ax.add_patch(rect2)
ax.text(x1, y1, f"Person (Conf: {probs[0,1]:.2f})", color='red', fontsize=12, bbox=dict(facecolor='white', alpha=0.5))

# Add custom legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color='yellow', lw=2, linestyle='dashed', label='Input Proposal (RoI)'),
                   Line2D([0], [0], color='red', lw=2, label='Network Refined Output')]
ax.legend(handles=legend_elements, loc='upper right')

plt.title("Fast R-CNN: Bounding Box Regression in Action!")
plt.axis('off')
plt.show()

